In [5]:
import pandas as pd
import re
from transformers import AutoTokenizer

In [6]:
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

data_dict = {
    "text": [
        "  The staff was very kind and attentive to my needs!!!  ",
        "The waiting time was too long, and the staff was rude. Visit us at http://hospitalreviews.com",
        "The doctor answered all my questions...but the facility was outdated.   ",
        "The nurse was compassionate & made me feel comfortable!! :) ",
        "I had to wait over an hour before being seen.  Unacceptable service! #frustrated",
        "The check-in process was smooth, but the doctor seemed rushed. Visit https://feedback.com",
        "Everyone I interacted with was professional and helpful.  "
    ],
    "label": ["positive", "negative", "neutral", "positive", "negative", "neutral", "positive"]
}

In [7]:
data = pd.DataFrame(data_dict)

def clean_text(text):
    text = text.lower()
    text = re.sub(r'http\S+', '', text)
    text = re.sub(r'[^\w\s]', '', text)
    text = re.sub(r'\s+', '', text).strip()
    return text

data["cleaned_text"] = data["text"].apply(clean_text)

label_map = {"positive": 0, "neutral": 1, "negative": 2}
data["label"] = data["label"].map(label_map)

data['tokenized'] = data['cleaned_text'].apply(lambda x: tokenizer.encode(x, add_special_tokens=True))
data['padded_tokenized'] = data['tokenized'].apply(
    lambda x: x + [tokenizer.pad_token_id] * (128 - len(x)) if len(x)< 128 else x[:128]
)

print(data[['cleaned_text', 'label', 'padded_tokenized']].head())

                                        cleaned_text  label  \
0           thestaffwasverykindandattentivetomyneeds      0   
1  thewaitingtimewastoolongandthestaffwasrudevisi...      2   
2  thedoctoransweredallmyquestionsbutthefacilityw...      1   
3      thenursewascompassionatemademefeelcomfortable      0   
4  ihadtowaitoveranhourbeforebeingseenunacceptabl...      2   

                                    padded_tokenized  
0  [101, 1996, 9153, 4246, 17311, 27900, 18824, 1...  
1  [101, 1996, 21547, 3436, 7292, 17311, 3406, 12...  
2  [101, 1996, 3527, 16761, 6962, 13777, 11960, 3...  
3  [101, 2059, 28393, 17311, 9006, 15194, 3258, 3...  
4  [101, 1045, 16102, 18790, 4886, 26525, 23169, ...  


In [8]:
from sklearn.model_selection import train_test_split

train_data, temp_data = train_test_split(data, test_size = 0.3, random_state= 42)
val_data, test_data = train_test_split(temp_data, test_size= 0.5, random_state= 42)

print(f"Training size: {len(train_data)}, Validation size: {len(val_data)}, Test data: {len(test_data)}")

Training size: 4, Validation size: 1, Test data: 2


In [9]:
from datasets import Dataset

train_dataset = Dataset.from_pandas(train_data)
val_dataset = Dataset.from_pandas(val_data)
test_dataset = Dataset.from_pandas(test_data)

def tokenize_function(examples):
    return tokenizer(examples["cleaned_text"], padding= "max_length", truncation= True, max_length= 128)

train_dataset = train_dataset.map(tokenize_function, batched= True)
val_dataset = val_dataset.map(tokenize_function, batched= True)
test_dataset = test_dataset.map(tokenize_function, batched= True)

train_dataset = train_dataset.remove_columns(["text", "cleaned_text"])
val_dataset = val_dataset.remove_columns(["text", "cleaned_text"])
test_dataset = test_dataset.remove_columns(["text", "cleaned_text"])

train_dataset = train_dataset.map(lambda x: {"label" : int(x["label"])})
val_dataset = val_dataset.map(lambda x: {"label" : int(x["label"])})
test_dataset = test_dataset.map(lambda x: {"lambda" : int(x["label"])})

print(train_dataset[0])

Map: 100%|██████████| 2/2 [00:00<00:00, 666.24 examples/s]

{'label': 1, 'tokenized': [101, 1996, 3527, 16761, 6962, 13777, 11960, 3363, 8029, 15500, 8496, 8569, 4779, 5369, 7011, 6895, 18605, 17311, 5833, 13701, 2094, 102], 'padded_tokenized': [101, 1996, 3527, 16761, 6962, 13777, 11960, 3363, 8029, 15500, 8496, 8569, 4779, 5369, 7011, 6895, 18605, 17311, 5833, 13701, 2094, 102, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], '__index_level_0__': 2, 'input_ids': [101, 1996, 3527, 16761, 6962, 13777, 11960, 3363, 8029, 15500, 8496, 8569, 4779, 5369, 7011, 6895, 18605, 17311, 5833, 13701, 2094, 102, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 

In [10]:
from transformers import AutoModelForSequenceClassification, TrainingArguments, Trainer

model = AutoModelForSequenceClassification.from_pretrained("bert-base-uncased", num_labels= 3)

training_args = TrainingArguments(
    learning_rate= 2e-5,
    per_device_train_batch_size = 16,
    num_train_epochs = 3,
    output_dir = "./results",
    evaluation_strategy = 'epoch',
    logging_strategy = 'epoch',
    logging_dir = './logs',
    save_strategy = "epoch",
    load_best_model_at_end = True
)

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
C:\Users\Thamotharan\AppData\Local\Programs\Python\Python310\lib\site-packages\transformers\training_args.py:1474: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead
  warnings.warn(


In [11]:
from transformers import Trainer

trainer = Trainer(
    model = model,
    args= training_args,
    train_dataset= train_dataset.with_format("torch", columns=["input_ids", "attention_mask", "label"]),
    eval_dataset= val_dataset.with_format("torch", columns=["input_ids", "attention_mask", "label"])
)

trainer.train()

Epoch,Training Loss,Validation Loss
1,1.111300,1.028831
2,1.042500,1.063060
3,1.088600,1.105643


Attempted to log scalar metric loss:
1.1113
Attempted to log scalar metric grad_norm:
6.187999248504639
Attempted to log scalar metric learning_rate:
1.3333333333333333e-05
Attempted to log scalar metric epoch:
1.0
Attempted to log scalar metric eval_loss:
1.0288314819335938
Attempted to log scalar metric eval_runtime:
0.1821
Attempted to log scalar metric eval_samples_per_second:
5.491
Attempted to log scalar metric eval_steps_per_second:
5.491
Attempted to log scalar metric epoch:
1.0
Attempted to log scalar metric loss:
1.0425
Attempted to log scalar metric grad_norm:
13.119780540466309
Attempted to log scalar metric learning_rate:
6.666666666666667e-06
Attempted to log scalar metric epoch:
2.0
Attempted to log scalar metric eval_loss:
1.06305992603302
Attempted to log scalar metric eval_runtime:
0.1838
Attempted to log scalar metric eval_samples_per_second:
5.44
Attempted to log scalar metric eval_steps_per_second:
5.44
Attempted to log scalar metric epoch:
2.0
Attempted to log sca

TrainOutput(global_step=3, training_loss=1.0808192888895671, metrics={'train_runtime': 10.4073, 'train_samples_per_second': 1.153, 'train_steps_per_second': 0.288, 'total_flos': 789340253184.0, 'train_loss': 1.0808192888895671, 'epoch': 3.0})

In [14]:
from sklearn.metrics import accuracy_score, f1_score

test_dataset = test_dataset.with_format("torch", columns=["input_ids", "attention_mask", "label"])
predictions = trainer.predict(test_dataset)
preds = predictions.predictions.argmax(-1)
labels = test_dataset["label"]

accuracy = accuracy_score(labels, preds)
f1 = f1_score(labels, preds, average="weighted")

print(f"Accuracy: {accuracy}, F1 accuracy: {f1}")

Accuracy: 0.5, F1 accuracy: 0.5


In [15]:
model.save_pretrained("./fine-tuned-bert")
tokenizer.save_pretrained("./fine-tuned-bert")
print("Model Saved Successfully")

Model Saved Successfully
